In [1]:
def hunt_db(input_search,**kwargs):
    
    #//**** Assign Default Values
    input_type='int'
    
    export = False
    
    display = {
        "items" : True,
    }
    
    #//*** Scan Kwargs and update defaults as needed
    for key,value in kwargs.items():
        if key == "items":
            display[key] = value
        
        if key == "input_type":
            input_type=value
        
        if key == "export_dict":
            export = True

    query = """
    SELECT * 
    FROM INFORMATION_SCHEMA.COLUMNS 
    """
    cursor.execute(query)
    results = cursor.fetchall()
    titleid_dict = {}
    #//*** Script ID 29807928
    #//*** Rundown ID 29620400
    for row in results:
        #print(row[2], row[3])
        if  row[7] == input_type:
            #print(row[2],row[3])
            loop_query = f"""
            SELECT count(*)
            FROM {row[2]}
            WHERE {row[3]}='{input_search}' 
            """
            cursor.execute(loop_query)
            loop_result = cursor.fetchall()[0][0]
            if loop_result > 0:
                print(row[2],row[3],loop_result)
                titleid_dict[row[2]] = row[3]
    
    if export:
        print("Exporting Table Dict")
        return titleid_dict
    
    for table,column in titleid_dict.items():
        if not display['items']:
            if table == 'items':
                print("**************************************")
                print("Skipping Rundown Creation Items.")
                print("Use items=True to display")
                print("**************************************")
                
                continue
            
        loop_query = f"""
        SELECT *
        FROM {table}
        WHERE {column}='{input_search}' 
        """
        cursor.execute(loop_query)
        loop_result = cursor.fetchall()
        print(f"{table} -- {column}")
        out=""
        for desc in cursor.description:
            out += f"{desc[0]}, "
        print(out)
        #print(desc)
        #print ([x for x in cursor.description])
        print("Table")
    

        
        for row in loop_result:

            out = str(len(row)) + ":( "
            
            for field in row:
                out += str(field).replace("&gt;",">").replace("&lt;","\n<").replace("&quot;","\"").replace("&#xd;&#xa;","\n").replace("&#xa;&#xa;","\n")
                out += ", "
            out += " )"
            print(out)

        print("=============")

In [2]:
def handleError(msg):
    print(msg)

In [3]:
#Connect to UNC path
#\\om-casf-dlbr06\MEDIA

In [2]:
#//*** Tips for converting to EXE
#//*** https://towardsdatascience.com/how-to-easily-convert-a-python-script-to-an-executable-file-exe-4966e253c7e9

#//*** pyinstaller --onefile music.py

import time,sys, threading, socket, binascii,  os, re
import vlc #pip install python-vlc
#import keyboard #pip install keyboard

#from pynput import keyboard #pip install pynput
#//*** Keyboard basics
#https://www.delftstack.com/howto/python/python-detect-keypress/
#//*** Threading Keyboard listener
#https://stackoverflow.com/questions/14043441/how-to-use-threads-to-get-input-from-keyboard-in-python-3
#//*** pyinstaller --onefile music.py -n kgo_digi_player_v1

#//*********************************************
#//*** SQL Section
#//*********************************************
#//*** This thread helped with the connection
#//***https://stackoverflow.com/questions/37692780/error-28000-login-failed-for-user-domain-user-with-pyodbc

#//**** Update the ODBC Driver
#//**** https://docs.microsoft.com/en-us/sql/connect/odbc/download-odbc-driver-for-sql-server?view=sql-server-ver15
import pyodbc
import datetime
import json
from datetime import date, timedelta
import pandas as pd
import numpy as np 
print(pyodbc.drivers())

#//*********************************************
#//*** SQL Setup/Initialization
#//*********************************************

#//*** Pandas Display Options
#//*** Maximize columns and rows displayed by pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Some other example server values are
# server = 'localhost\sqlexpress' # for a named instance
# server = 'myserver,port' # to specify an alternate port

#//*** These Values sre shifted to rp.config
#server = 'tcp:OM-CASF-DB01' 
#server = 'OM-CASF-DLSQL' 
# server = '10.218.97.2'
#database = 'DaletDB' 


#//*********************************************
#//*********************************************
#//*********************************************





with open('./ignore_folder/misc.json') as f:
    data = json.loads(f.read())

username = data["user"] 
password = data["password"]
del data

print(username + ":" + password)


folder_path = "./music/1_0_0 - 3pm Playlist"

music_path = "./music/"

config_filename = "rp.config"
#//*** Set Default host IP 
h_name = socket.gethostname()
HOST = socket.gethostbyname(h_name)
PORTS = [10522]
rundown_names = ["5PM Weekday"]
playout_paths = ["\\\\om-casf-dlbr06\\MEDIA"]


#//*** Clear the Screen on each action. False aids in debugging.
clear_screen = True
fullscreen = True

action_queue = []
rp = {

    #//*** Build Playlist
    "playlist" : [],
    "play_counter" : 0,
    "display_name" : "",
    "playing" : False,
    "quit" : False,
    "action" : None,
    #"socket" : socket.socket(socket.AF_INET, socket.SOCK_STREAM),
    "addr" : [],
    "selected_cut" : None,
    "cut_index" : None,
    "cut" : {},
    "active_ports" : []
    
}   


#//*** Process Config File
#//*** Allows for Comments after # sign
#//*** All values are explictly defined.
if os.path.exists(config_filename):
    print("Process Config File")
    with open(config_filename) as f:
        for line in f:
            
            #//*** strip out everything after # comment
            if "#" in line:
                line = line.split("#")[0]


            #//*** Look for lines with values:
            if "=" in line:

                key,value = line.split("=")

                #//*** Strip all whitespace from key & value
                key = key.strip()
                value = value.strip().replace("\n","")

                print(f">{key}<")
                print(f">{value}<")

                if key == "server":
                    server = value

                if key == "database":
                    database = value

# Some other example server values are
# server = 'localhost\sqlexpress' # for a named instance
# server = 'myserver,port' # to specify an alternate port
#server = 'tcp:OM-CASF-DB01' 
server = 'OM-CASF-DLSQL' 
# server = '10.218.97.2'
database = 'DaletDB' 

"""
with open('./ignore_folder/misc.json') as f:
    data = json.loads(f.read())

username = data["user"] 
password = data["password"]

username = "user"
password = "pass"
del data
"""

#cnxn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
cnxn = pyodbc.connect('Trusted_Connection=yes;DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
cursor = cnxn.cursor()

print("next")


['SQL Server', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client 11.0']
$kgotvscripts:ABC7news
next


In [5]:
#<Value Type="Story" Id="44555807" Name="530P KNO + CASEY CGs" ParentId="43675" LocalSite="" Site="KGO" ItemId="44513775:1000181612:5549966"/>

#//*** StoryID
print("=========================")
print("STORY ID")
print("=========================")
hunt_db("44555807")
#//*** Block ID
print("=========================")
print("ITEM ID")
print("=========================")
hunt_db("5549966")
print("=========================")
print("BLOCK ID")
print("=========================")
hunt_db("1000181612")
#hunt_db("44513775")

STORY ID
Titles_ExFields Title_id 1
TitleDataDictInteger TitleId 43
TitleDataDictInteger NumValue 1
TitleDataDictString TitleId 32
TitleDataDictDateTime TitleId 2
TitleDataDictText TitleId 2
Versions OrginalTitleId 1
titles_extend title_id 1
ItemsToBeIndexed Item_id 1
spots title_id 1
titles_in_class title_id 1
objects_in_titles Title_id 2
titles_references title_id 1
titles_mix title_id 1
CreatedModifiedItems Item_id 1
StoryContent TitleId 1
ObjectStatus ObjectId 1
titles title_id 1
StatusLog ObjectId 3
Titles_ExFields -- Title_id
Title_id, Language_id, Topic, VersionComment, 
Table
4:( 44555807, 99,  , None,  )
TitleDataDictInteger -- NumValue
DataField_Id, TitleId, ValueOrder, NumValue, 
Table
4:( 682, 44556182, 1, 44555807,  )
TitleDataDictString -- TitleId
DataField_Id, TitleId, ValueOrder, StrValue, 
Table
4:( 518, 44555807, 1, Not Specified,  )
4:( 519, 44555807, 1, Not Specified,  )
4:( 1173, 44555807, 1, FORMAT,  )
4:( 1178, 44555807, 1, --,  )
4:( 1213, 44555807, 1, web,  )
4

items block_id 17
spots block_id 15
texts block_id 2
block_item child_block_id 1
blocks block_id 1
items -- block_id
block_id, item_id, item_type, item_reference, scheduled_time, computed_time, item_comment, item_order, radio_id, clock_id, use_manual_duration, 
Table
11:( 1000181612, 5536035, 3,  , 2023-07-12 23:59:59.940000, 2023-07-12 23:59:59.940000, <Text blankLineColor.l="16777215" blankLineTransparent.b="false" enteredDateTime.td="2023-01-11T18:27:49.365" mode.l="0" startMode.l="1"/>, 10000, 1132, 44513775, 0,  )
11:( 1000181612, 5536038, 1,  , 2023-07-12 23:59:59.940000, 2023-07-12 23:59:59.940000, <Story copyEmbeddedTitlesInApplyPattern.l="1" copyTitleInApplyPattern.l="1" enteredDateTime.td="2021-01-25T17:55:01.770" mode.l="0" orderPageNumber.l="11" startMode.l="1" tp.l="1"><custom2 t="ws">IW</custom2><custom3 t="ws">$$$$1.0</custom3><pageNumber t="ws">F11</pageNumber></Story>, 40000, 1132, 44513775, 0,  )
11:( 1000181612, 5536039, 1,  , 2023-07-12 23:59:59.940000, 2023-07-12 2

Query Database for Today's rundowns

In [9]:
#Get Current Rundown IDs from resourceReservations, ClockID and Name in Content

#//*** Get Rundown ID of Shows based on Rundown Name which needs to be a Unique and Exact Match. There can be trouble if there are multiple Rundowns wit the same name. 
#//*** We Should Probably take that into account when welaunch the Final Product

todayStart = str(date.today())
todayEnd = str(date.today() + timedelta(days=1))
date_range = f"'{todayStart}T00:00:00' AND '{todayEnd}T00:00:00'"

rundown_target = rundown_names[0]

print("StartDate:"+todayStart)
print("endDate:"+todayEnd)
print(date_range)



query = f"""
SELECT *
FROM resourceReservations 
WHERE startTime BETWEEN {date_range}
AND content LIKE '%{rundown_target}%'
"""
#WHERE stationId = '1132'

 

cursor.execute(query)
results = cursor.fetchall()

print(len(results))
df = pd.read_sql(query,cnxn)
df



StartDate:2023-07-07
endDate:2023-07-08
'2023-07-07T00:00:00' AND '2023-07-08T00:00:00'
1


C:\Users\stonk013\AppData\Local\Temp\ipykernel_8812\1358546197.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query,cnxn)


,reservationId,clockId,studioId,stationId,reservedBy,startTime,endTime,status,content,ReservationMode,MediaProfileId,MetadataFormatId
0,136663,44435540,71.0,1132,STUDIO_ID_71,2023-07-07 23:59:59.940,2023-07-08 00:57:15.903,0,"<ClockAssignment><descriptionOfUsage t=""ws"">CL...",0,None,None


In [7]:


rp = {
    "rundown" : {
        "data_fields" : ["clockId","content","startTime"],
        "data_index" : {},
        "data" : {},
        "name_replace_1" : '<name t="ws">CLOCK ',
        "name_replace_2" : '</name>',
        "name":"XXX PLACEHOLDER XXX",
        "id" : None,
    }
}


#//*** Get the Column Names From the Cursor
metas = cursor.columns(table='resourceReservations') 

#//*** Build Column Indexes and Store in rp, Rundown
for x,meta in enumerate(metas):    
    if meta[3] in rp['rundown']["data_fields"]:
        rp['rundown']["data_index"][meta[3]] = x

print(rp["rundown"]["data_index"])


#//*** Currently assuming there is only one Rundown with this name, We should account for multiple.

if len(results) == 1:
    result = results[0]      
else:
    result = None
    handleError(f"EC 1: Returned Multiple Rundowns with name {rundown_target}. Expected Answer is 1.\nNeed Code to deal with edge case. Expect a Crash after this.")

for field,index in rp["rundown"]["data_index"].items():
    rp["rundown"]["data"][field] = result[index]

for key,value in rp["rundown"]["data"].items():
    print(key,value)

content = rp["rundown"]["data"]["content"]

try:
    content = re.findall('<name t="ws">.+</name>',content)[0]
except:
    pass

#//*** Get Rundown Name From Content. Must Strip XML fields from Content
rp["rundown"]["name"] = content.replace(rp["rundown"]["name_replace_1"],"").replace(rp["rundown"]["name_replace_2"],"")

rp["rundown"]["id"] = rp["rundown"]["data"]["clockId"]

print(rp["rundown"]["id"],rp["rundown"]["name"])


{'clockId': 1, 'startTime': 5, 'content': 8}
clockId 44435540
startTime 2023-07-07 23:59:59.940000
content <ClockAssignment><descriptionOfUsage t="ws">CLOCK 5PM Weekday assigned to STUDIO_ID_71</descriptionOfUsage><history t="lt"><HistoryElement at.tcd="U2023-07-05T17:35:41.347 TC:60000:1001" status.l="0"><description t="ws">CREATED</description></HistoryElement></history><name t="ws">CLOCK 5PM Weekday</name></ClockAssignment>
44435540 5PM Weekday


In [12]:
#//*** Get Ordered Scripts from Items by clock_id (rundown id). Breaks will be associated will Specific Clip Names and Handled with the VLC player 
#//*** Get Script IDs from items clock_id

#//item_type 1 = Scripts
#//item_type 3 = Blank Line
#//item_type 5 = Block


query = f"""
SELECT *
FROM items 
WHERE clock_id = '{rp["rundown"]["id"]}'
AND item_type = 5

"""
#ORDER BY item_order

cursor.execute(query)
results = cursor.fetchall()

print(len(results))

items_df = pd.read_sql(query,cnxn)
items_df

8


C:\Users\stonk013\AppData\Local\Temp\ipykernel_8812\3098457990.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  items_df = pd.read_sql(query,cnxn)


,block_id,item_id,item_type,item_reference,scheduled_time,computed_time,item_comment,item_order,radio_id,clock_id,use_manual_duration
0,1000180883,5509644,5,,2023-07-07 23:59:59.940,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",40000,1132,44435540,0
1,1000180883,5509661,5,,2023-07-08 00:00:00.940,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",50000,1132,44435540,0
2,1000180883,5509668,5,,2023-07-08 00:00:01.940,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",60000,1132,44435540,0
3,1000180883,5509677,5,,2023-07-08 00:00:02.943,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",70000,1132,44435540,0
4,1000180883,5509696,5,,2023-07-08 00:00:03.007,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",80000,1132,44435540,1
5,1000180883,5509712,5,,2023-07-08 00:05:03.007,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",90000,1132,44435540,1
6,1000180883,5509720,5,,2023-07-08 00:10:03.977,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",100000,1132,44435540,1
7,1000180883,5509727,5,,2023-07-08 00:10:03.007,2023-07-07 23:59:59.940,"<Block blockStartMode.l=""1"" enteredDateTime.td...",120000,1132,44435540,1


In [6]:
import numpy as np 
item_cols = ['block_id','item_id','item_order', 'item_comment']
items_df = items_df[item_cols]

#//**** Build Ordered list of Item IDs.
#//**** Start with ORdered List of Blocks using Numpy Sort
for block in np.sort(items_df['block_id'].unique()):
    
    df = items_df[items_df["block_id"] == block].sort_values('item_order')
    print("==========")
    print(block)
    print("==========")
    for row in df.iterrows():
        print(row[1])
        print(row[1]['block_id'])
        print(row[1]['item_comment'])





1000180745
block_id                                               1000180745
item_id                                                   5503571
item_order                                                  10000
item_comment    <Story copyEmbeddedTitlesInApplyPattern.l="1" ...
Name: 0, dtype: object
1000180745
<Story copyEmbeddedTitlesInApplyPattern.l="1" copyTitleInApplyPattern.l="1" enteredDateTime.td="2021-03-25T02:36:27.079" hold.l="0" mode.l="0" orderPageNumber.l="1" startMode.l="1" tp.l="1"><custom3 t="ws">6/11/21updated</custom3><pageNumber t="ws">1</pageNumber></Story>
block_id                                               1000180745
item_id                                                   5503572
item_order                                                  20000
item_comment    <Story enteredDateTime.td="2023-02-21T18:23:17...
Name: 1, dtype: object
1000180745
<Story enteredDateTime.td="2023-02-21T18:23:17.103" mode.l="0" orderPageNumber.l="2" startMode.l="1" tp.l="1"><pageNumber

In [7]:
results

[(1000180745, 5503571, 1, ' ', datetime.datetime(2023, 7, 6, 23, 59, 59, 940000), datetime.datetime(2023, 7, 6, 23, 59, 59, 940000), '<Story copyEmbeddedTitlesInApplyPattern.l="1" copyTitleInApplyPattern.l="1" enteredDateTime.td="2021-03-25T02:36:27.079" hold.l="0" mode.l="0" orderPageNumber.l="1" startMode.l="1" tp.l="1"><custom3 t="ws">6/11/21updated</custom3><pageNumber t="ws">1</pageNumber></Story>', 10000, 1132, 44417203, 0),
 (1000180745, 5503572, 1, ' ', datetime.datetime(2023, 7, 6, 23, 59, 59, 940000), datetime.datetime(2023, 7, 6, 23, 59, 59, 940000), '<Story enteredDateTime.td="2023-02-21T18:23:17.103" mode.l="0" orderPageNumber.l="2" startMode.l="1" tp.l="1"><pageNumber t="ws">2</pageNumber></Story>', 20000, 1132, 44417203, 0),
 (1000180745, 5503573, 1, ' ', datetime.datetime(2023, 7, 6, 23, 59, 59, 940000), datetime.datetime(2023, 7, 6, 23, 59, 59, 940000), '<Story copyEmbeddedTitlesInApplyPattern.l="1" copyTitleInApplyPattern.l="1" enteredDateTime.td="2021-01-27T22:11:20.

In [8]:
#hunt_db("5503575",items=True)
#hunt_db("1000180609", items=False)
#hunt_db("5508751", items=False)




In [9]:
title_id = "5503575"

query = f"""
SELECT title_id
FROM spots
WHERE item_id = '{title_id}'
"""
#ORDER BY item_order

cursor.execute(query)
results = cursor.fetchall()

if len(results) > 0:
    story_id = results[0][0]
    print(story_id)
else:
    print("Need some error handling in the unlikely event we get more than one story back")
    pass



44417223


In [17]:
#hunt_db("5503657") #//*** BlockID value
hunt_db("5515832") #//*** Block STory Id Value
#hunt_db("44456183")#//*** Story ID Example
#hunt_db("44456917")#//*** Video MOS Object Example



items item_id 1
spots item_id 1
items -- item_id
block_id, item_id, item_type, item_reference, scheduled_time, computed_time, item_comment, item_order, radio_id, clock_id, use_manual_duration, 
Table
11:( 1000180749, 5515832, 1,  , 2023-07-06 23:59:59.937000, 2023-07-06 23:59:59.937000, <Story mode.l="0" orderPageNumber.l="4" plannedDuration.tc="P0.000S" startMode.l="1" tp.l="1"><custom3 t="ws">5emo</custom3><pageNumber t="ws">D4</pageNumber></Story>, 35000, 1132, 44417203, 0,  )
spots -- item_id
block_id, item_id, title_id, mix_type, volume, begin_offset, end_offset, fade_in, fade_out, crossfade, 
Table
10:( 1000180749, 5515832, 44455801, 0, 0, 0, 0, 0, 0, 0,  )


In [11]:
#//*** Get Story Content, this is where we get MOS Info


story_id = "44453964"

query = f"""
SELECT StoryXml
FROM StoryContent
WHERE TitleId = '{story_id}'
"""

cursor.execute(query)
results = cursor.fetchall()

content = str(results[0])

#content.replace("\\\\par&#xd;&#xa;","\n")
raw_ids = []
mos_xml = re.findall('<EmbeddedObject.+?</EmbeddedObject>',content)
for raw in mos_xml:
    
    if "duration.tc=" in raw:
        #//*** This is a video Event (Most Likely)
        print(raw)
        
        #//*** This absolutely should be here
        if "ObjectId.l=" in raw: 
            mos_id = re.findall('ObjectId.l=".+?"',raw)
            
            #//*** Cleanup MOS ID. 
            if len(mos_id) == 1:
                mos_id = mos_id[0].replace('ObjectId.l=',"").replace('"',"")
                print(mos_id)
                raw_ids.append(mos_id)
            else:
                print("Throw an error in Handling the MOS ID regex Cleanup")
                pass


<EmbeddedObject ObjectId.l="44453963" ObjectType.l="2" PositionInText.l="1" idInStory.l="1"><itemSpot t="t"><Spot beginOffset.tc="P0.000S" beginOffsetEnd.l="0" duration.tc="P0.000S" endOffset.tc="P2:18.339S" endOffsetBegin.l="1"/></itemSpot></EmbeddedObject>
44453963


In [15]:
for raw_id in raw_ids:
    print(raw_id)
    #titles_references title_id 1
     
    query = f"""
    SELECT *
    FROM titles_references  
    WHERE title_id  = '{raw_id}'
    """

    cursor.execute(query)
    results = cursor.fetchall()
    
    print(results)

44453963
[(44453963, 1, 'KGO17203468')]


In [13]:
table_dict = hunt_db("44453963",export_dict=True)
print(table_dict)

Titles_ExFields Title_id 1
TitleDataDictInteger TitleId 35
TitleDataDictString TitleId 30
TitleDataDictDateTime TitleId 2
TitleDataDictText TitleId 2
TitleDataDictTitle TitleId 1
TitleDataDictTitle EmbeddedTitle 1
TitleDataDictPeriod TitleId 1
TitleDataDictTimeCode TitleId 2
TitleDataDictTimeCodeDate TitleId 1
TitleCreationForms TitleId 1
titles_extend title_id 1
ItemsToBeIndexed Item_id 1
TitleLastUsedForms TitleId 1
Locators ObjectId 35
LocatorThumbnails ObjectId 66
titles_in_class title_id 1
objects_in_titles Object_id 15
GenealogyRenderedSegments RenderedTitleId 30
GenealogySegmentAMfields RenderedTitleId 1
titles_references title_id 1
titles_mix title_id 1
CreatedModifiedItems Item_id 1
Title_Files Title_id 6
ObjectStatus ObjectId 1
titles title_id 1
StatusLog ObjectId 2
Exporting Table Dict
{'Titles_ExFields': 'Title_id', 'TitleDataDictInteger': 'TitleId', 'TitleDataDictString': 'TitleId', 'TitleDataDictDateTime': 'TitleId', 'TitleDataDictText': 'TitleId', 'TitleDataDictTitle': '

In [14]:
#//**** Reference for getting around IO datarate Issues in Notebook
for table,col in table_dict.items():

    query = f"""
    SELECT *
    FROM {table}
    WHERE {col} = '{raw_id}'
    """

    cursor.execute(query)
    results = cursor.fetchall()
    
    print("========================")
    print(f"Table: {table}")
    print("========================")
    print(len(results),len(results[0]))
    time.sleep(1)
    for row in results:
        print(row)
    

Table: Titles_ExFields
1 4
(44453963, 99, None, None)
Table: TitleDataDictInteger
35 4
(291, 44453963, 1, 0)
(647, 44453963, 1, 1)
(648, 44453963, 1, 1)
(649, 44453963, 1, 1)
(650, 44453963, 1, 1)
(651, 44453963, 1, 1)
(652, 44453963, 1, 1)
(653, 44453963, 1, 1)
(654, 44453963, 1, 1)
(655, 44453963, 1, 1)
(656, 44453963, 1, 1)
(657, 44453963, 1, 1)
(658, 44453963, 1, 1)
(659, 44453963, 1, 1)
(660, 44453963, 1, 1)
(661, 44453963, 1, 1)
(662, 44453963, 1, 1)
(663, 44453963, 1, 1)
(664, 44453963, 1, 1)
(665, 44453963, 1, 1)
(666, 44453963, 1, 1)
(667, 44453963, 1, 1)
(668, 44453963, 1, 1)
(669, 44453963, 1, 1)
(670, 44453963, 1, 1)
(671, 44453963, 1, 1)
(672, 44453963, 1, 1)
(673, 44453963, 1, 1)
(674, 44453963, 1, 1)
(675, 44453963, 1, 1)
(676, 44453963, 1, 1)
(677, 44453963, 1, 1)
(678, 44453963, 1, 1)
(1307, 44453963, 1, 0)
(1466, 44453963, 1, 0)
Table: TitleDataDictString
30 4
(51, 44453963, 1, 'VALID')
(286, 44453963, 1, 'Not Specified')
(518, 44453963, 1, 'Not Specified')
(519, 4445

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



(35678, 44453963, 1, None)
Table: objects_in_titles
15 5
(44453963, 2, 44452951, 0, -1)
(44453963, 2, 44453964, 0, -1)
(44453963, 2, 44457277, 0, -1)
(44453963, 2, 44457334, 0, -1)
(44453963, 2, 44457499, 0, -1)
(44453963, 2, 44457676, 0, -1)
(44453963, 2, 44457681, 0, -1)
(44453963, 2, 44457682, 0, -1)
(44453963, 2, 44458056, 0, -1)
(44453963, 2, 44458064, 0, -1)
(44453963, 2, 44458230, 0, -1)
(44453963, 2, 44458231, 0, -1)
(44453963, 2, 44458232, 0, -1)
(44453963, 2, 44459185, 0, -1)
(44453963, 2, 44459359, 0, -1)
Table: GenealogyRenderedSegments
30 11
(44453963, 1, 0, 5672, 44455462, 1, 2150615, 2156287, 60000, 1001, None)
(44453963, 2, 16366, 20887, 44455462, 1, 373122, 377643, 60000, 1001, None)
(44453963, 3, 20653, 25642, 44455462, 1, 1078660, 1083649, 60000, 1001, None)
(44453963, 4, 25408, 27343, 44455462, 1, 1699230, 1701166, 60000, 1001, None)
(44453963, 5, 27360, 29245, 44455462, 1, 2279510, 2281395, 60000, 1001, None)
(44453963, 6, 29262, 32332, 44455462, 1, 1620435, 162350

In [14]:

 
field = "Locators" 
col = "ObjectId"

query = f"""
SELECT *
FROM {field}
WHERE {col} = '{raw_id}'
"""

cursor.execute(query)
results = cursor.fetchall()

print(results)


field = "titles" 
col = "title_id"

query = f"""
SELECT *
FROM {field}
WHERE {col} = '{raw_id}'
"""

cursor.execute(query)
results = cursor.fetchall()

print(results)


field = "Titles_ExFields" 
col = "Title_id"

query = f"""
SELECT *
FROM {field}
WHERE {col} = '{raw_id}'
"""

cursor.execute(query)
results = cursor.fetchall()

print(results)


[(34802643, 44453963, 1, 'KEY-FRAME', 22939, -1, datetime.datetime(1970, 1, 1, 0, 0), datetime.datetime(1970, 1, 1, 0, 0), 0, 'P22.939583S TC:60000:1001', 'null', ' '), (34802664, 44453963, 1, 'KEY-FRAME', 29362, -1, datetime.datetime(1970, 1, 1, 0, 0), datetime.datetime(1970, 1, 1, 0, 0), 0, 'P29.362666S TC:60000:1001', 'null', ' '), (34802666, 44453963, 1, 'KEY-FRAME', 31014, -1, datetime.datetime(1970, 1, 1, 0, 0), datetime.datetime(1970, 1, 1, 0, 0), 0, 'P31.014316S TC:60000:1001', 'null', ' '), (34799757, 44453963, 1, 'POSTER-FRAME', 6006, -1, datetime.datetime(1970, 1, 1, 0, 0), datetime.datetime(1970, 1, 1, 0, 0), 0, None, None, None), (34799693, 44453963, 1, 'SOURCE-MATERIAL', 0, 5672, datetime.datetime(1970, 1, 1, 0, 0), datetime.datetime(1970, 1, 1, 0, 0), 0, None, None, ''), (34799694, 44453963, 1, 'SOURCE-MATERIAL', 16366, 20887, datetime.datetime(1970, 1, 1, 0, 0), datetime.datetime(1970, 1, 1, 0, 0), 0, None, None, ''), (34799695, 44453963, 1, 'SOURCE-MATERIAL', 20653, 25

In [32]:


query = f"""
SELECT CONVERT(DATE, GETDATE()) 
      AS 'CURRENT_TIME using GETDATE()'
"""

#cursor.execute(query)
#results = cursor.fetchall()



In [18]:
#//*** Dump Rundowns into Pandas

title_cols = ['title_id','title_type_id','title','duration','start_date']

day_rundown = pd.read_sql(query,cnxn)


target_rundown =rundown_names[0]
target_rundown

day_rundown


C:\Users\stonk013\AppData\Local\Temp\ipykernel_10444\1376185982.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  day_rundown = pd.read_sql(query,cnxn)


,CURRENT_TIME using GETDATE()
0,2023-07-05


In [5]:
result_rundown = day_rundown[day_rundown["title"] ==  target_rundown]
result_rundown 

,title_id,title_type_id,title_interface_id,user_id,title,interpret,author,client,duration,is_online,is_recorded,record_date,start_date,end_date,kill_date,audio_duration,use_manual_duration,soundfile_id,year,keywords,site_origin,no_overwrite,is_rotational_cart,replacement,last_words,is_opener,weight,beats_pm,master_record,compagny_id,compagny_disp_name,album_disp_name,package_id,album_id,song_id,day_part_rest_id,keep_date
15,44417203,25,0,0,5PM Weekday,,KGO DaletService,,3435965,1,0,2023-07-04 17:33:55.120,2023-07-04 17:33:55.120,9999-12-31 08:00:00,9999-12-31 08:00:00,0,0,0,2023,,0,0,0,,,0,100,-1,None,None,,,None,None,None,None,9999-12-31 08:00:00


In [6]:
#//*** Length Should be 1. Let's assume it's 1 or quit or do something
len(result_rundown)

if len(result_rundown) > 1:
    print("We have More Results than 1, Sounds like a multiple rundown Issue")
elif len(result_rundown) == 0:
    print("No Results Found, Handle Some Sort of ERROR / LOGIC PLANNING")

list(result_rundown["title_id"])[0]

target_rundown_id = list(result_rundown["title_id"])[0]
target_rundown_id

44417203

In [31]:
hunt_db(target_rundown_id,items=False)


Titles_ExFields Title_id 1
TitleDataDictInteger TitleId 33
TitleDataDictString TitleId 28
TitleDataDictDateTime TitleId 2
TitleDataDictText TitleId 2
items clock_id 129
resourceReservations clockId 1
titles_extend title_id 1
ItemsToBeIndexed Item_id 1
titles_in_class title_id 1
titles_references title_id 1
titles_mix title_id 1
CreatedModifiedItems Item_id 1
clocks clock_id 1
ObjectStatus ObjectId 1
titles title_id 1
StatusLog ObjectId 1
Titles_ExFields -- Title_id
Title_id, Language_id, Topic, VersionComment, 
Table
( 44417203, 99, None, None,  )
TitleDataDictInteger -- TitleId
DataField_Id, TitleId, ValueOrder, NumValue, 
Table
( 647, 44417203, 1, 1,  )
( 648, 44417203, 1, 1,  )
( 649, 44417203, 1, 1,  )
( 650, 44417203, 1, 1,  )
( 651, 44417203, 1, 1,  )
( 652, 44417203, 1, 1,  )
( 653, 44417203, 1, 1,  )
( 654, 44417203, 1, 1,  )
( 655, 44417203, 1, 1,  )
( 656, 44417203, 1, 1,  )
( 657, 44417203, 1, 1,  )
( 658, 44417203, 1, 1,  )
( 659, 44417203, 1, 1,  )
( 660, 44417203, 1, 1,  

In [ ]:
#Search Titles by Date and Rundown Name
#--> Generate title_id (Rundown ID)
#search Clocks, clock_id with Rundown ID --> Block ID